# 📄 Layout Detection (DocLayout-YOLO) + OCR (Tesseract)

### Pipeline
```
PDF page → image → DocLayout-YOLO → bboxes + class labels
         → filter TEXT classes only → crop each bbox → Tesseract → text
```

**DocLayout-YOLO class labels (DocStructBench model):**
| ID | Label | OCR? |
|----|-------|------|
| 0 | title | ✅ |
| 1 | plain text | ✅ |
| 2 | abandon (header/footer/noise) | ❌ |
| 3 | figure | ❌ |
| 4 | figure_caption | ✅ |
| 5 | table | ❌ |
| 6 | table_caption | ✅ |
| 7 | table_footnote | ✅ |
| 8 | isolate_formula | ❌ |
| 9 | formula_caption | ✅ |

**Tesseract PSM modes used per region type:**
- `title` → PSM 7 (single line)
- `plain text`, `figure_caption`, `table_caption`, `table_footnote`, `formula_caption` → PSM 6 (uniform block of text)

> Enable GPU: **Runtime → Change runtime type → T4 GPU**

## Step 1 — Install Dependencies

In [ ]:
# Tesseract engine
!apt-get install -y tesseract-ocr -q
!apt-get install -y tesseract-ocr-eng -q   # English language pack

# Python wrapper
!pip install pytesseract -q

# DocLayout-YOLO
!pip install doclayout-yolo -q
!pip install huggingface_hub -q

# PDF rendering
!pip install pdf2image -q
!apt-get install -y poppler-utils -q

# Verify tesseract installed
!tesseract --version

print('\n✅ Done!')

## Step 2 — Imports

In [ ]:
import random
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import pytesseract
from PIL import Image
from pdf2image import convert_from_path
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files

import warnings; warnings.filterwarnings('ignore')
print('✅ Imports OK')

## Step 3 — Load DocLayout-YOLO

In [ ]:
print('Downloading DocLayout-YOLO DocStructBench weights...')
model_path = hf_hub_download(
    repo_id='juliozhao/DocLayout-YOLO-DocStructBench',
    filename='doclayout_yolo_docstructbench_imgsz1024.pt'
)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
layout_model = YOLOv10(model_path)
print(f'✅ DocLayout-YOLO ready  (device: {DEVICE})')
print('✅ Tesseract ready (no model loading needed)')

## Step 4 — Upload PDFs

In [ ]:
uploaded = files.upload()
upload_dir = Path('/content/pdfs'); upload_dir.mkdir(exist_ok=True)

pdf_paths = []
for fname, content in uploaded.items():
    if not fname.lower().endswith('.pdf'):
        print(f'⚠️  Skipping {fname}'); continue
    p = upload_dir / fname
    p.write_bytes(content)
    pdf_paths.append(str(p))
    print(f'  ✅ {fname}  ({len(content)/1024:.1f} KB)')

print(f'\n{len(pdf_paths)} PDF(s) ready')

## Step 5 — Core Helpers

In [ ]:
DPI = 200

ID_TO_NAME = {
    0: 'title',
    1: 'plain text',
    2: 'abandon',
    3: 'figure',
    4: 'figure_caption',
    5: 'table',
    6: 'table_caption',
    7: 'table_footnote',
    8: 'isolate_formula',
    9: 'formula_caption'
}

TEXT_CLASSES = {'title', 'plain text', 'figure_caption',
                'table_caption', 'table_footnote', 'formula_caption'}

COLOURS = {
    'title':           'blue',
    'plain text':      'green',
    'abandon':         'gray',
    'figure':          'brown',
    'figure_caption':  'orange',
    'table':           'red',
    'table_caption':   'orange',
    'table_footnote':  'purple',
    'isolate_formula': 'magenta',
    'formula_caption': 'pink',
}

# Tesseract PSM per region type
# PSM 6 = assume uniform block of text (best for paragraphs)
# PSM 7 = treat image as single text line (best for titles)
# PSM 11 = sparse text, no particular order (fallback for noisy regions)
PSM_MAP = {
    'title':           7,
    'plain text':      6,
    'figure_caption':  6,
    'table_caption':   6,
    'table_footnote':  6,
    'formula_caption': 6,
}


# ── 1. DocLayout-YOLO inference ────────────────────────────────
def run_layout_detection(page_img: Image.Image, conf: float = 0.2) -> list:
    img_array = np.array(page_img)
    results = layout_model.predict(
        img_array, imgsz=1024, conf=conf,
        device=DEVICE, verbose=False
    )[0]

    detections = []
    boxes   = results.boxes.xyxy.cpu().numpy()
    classes = results.boxes.cls.cpu().numpy()
    scores  = results.boxes.conf.cpu().numpy()

    for box, cls_id, score in zip(boxes, classes, scores):
        cls_id = int(cls_id)
        detections.append({
            'label':      ID_TO_NAME.get(cls_id, f'class_{cls_id}'),
            'class_id':   cls_id,
            'confidence': float(score),
            'xyxy':       [int(v) for v in box]
        })

    detections.sort(key=lambda d: d['xyxy'][1])   # top → bottom
    return detections


# ── 2. Crop a detection bbox ───────────────────────────────────
def crop_detection(page_img: Image.Image, det: dict, padding: int = 6) -> Image.Image | None:
    x0, y0, x1, y1 = det['xyxy']
    W, H = page_img.size
    x0 = max(0, x0 - padding);  x1 = min(W, x1 + padding)
    y0 = max(0, y0 - padding);  y1 = min(H, y1 + padding)
    if x1 <= x0 or y1 <= y0:
        return None
    return page_img.crop((x0, y0, x1, y1))


# ── 3. Tesseract OCR on a single crop ─────────────────────────
def tesseract_ocr(crop: Image.Image, label: str) -> str:
    """
    Run Tesseract on a cropped region.
    - Upscale small crops to improve accuracy (Tesseract likes ≥300 DPI equivalent)
    - Use label-specific PSM for best results
    """
    # Upscale if crop is small — Tesseract accuracy drops below ~30px height
    w, h = crop.size
    if h < 60:
        scale = max(2, 60 // h)
        crop = crop.resize((w * scale, h * scale), Image.LANCZOS)

    psm = PSM_MAP.get(label, 6)
    config = f'--oem 3 --psm {psm} -l eng'

    text = pytesseract.image_to_string(crop, config=config)
    return text.strip()


# ── 4. Run Tesseract across all text detections ────────────────
def ocr_detections(page_img: Image.Image, detections: list) -> list:
    """
    Returns list of (label, text) for TEXT_CLASS regions, top-to-bottom.
    """
    lines = []
    text_dets = [d for d in detections if d['label'] in TEXT_CLASSES]
    for det in text_dets:
        crop = crop_detection(page_img, det)
        if crop is None:
            continue
        text = tesseract_ocr(crop, det['label'])
        if text:
            lines.append((det['label'], text))
    return lines


# ── 5. Visualise bboxes ────────────────────────────────────────
def visualise_detections(page_img: Image.Image, detections: list, title: str = '') -> None:
    fig, ax = plt.subplots(1, 1, figsize=(11, 15))
    ax.imshow(page_img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

    for det in detections:
        x0, y0, x1, y1 = det['xyxy']
        label = det['label']
        color = COLOURS.get(label, 'gray')
        conf  = det['confidence']
        rect  = patches.Rectangle(
            (x0, y0), x1-x0, y1-y0,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x0, max(y0-4, 0), f'{label} ({conf:.2f})',
                fontsize=7, color=color,
                bbox=dict(facecolor='white', alpha=0.5, pad=1, linewidth=0),
                clip_on=True)

    handles = [patches.Patch(color=c, label=l) for l, c in COLOURS.items()]
    ax.legend(handles=handles, loc='upper right', fontsize=7, framealpha=0.85)
    plt.tight_layout()
    plt.show()


print('✅ Helpers defined')

## Step 6 — Process Each PDF: Layout → Visualise → OCR

Target pages:
- `Researchpaper_KAI` → page **6**
- `apple 10k KAI` → page **38**
- `Laptopcatalogue_KAI` → page **5**
- Others → random

In [ ]:
PAGE_MAP = {
    'researchpaper_kai':    6,
    'apple 10k kai':        38,
    'laptopcatalogue_kai':  5,
}


for pdf_path in pdf_paths:
    filename   = Path(pdf_path).name
    stem_lower = Path(pdf_path).stem.lower()

    print(f'\n{"="*65}')
    print(f'📄  PDF : {filename}')

    # ── A. Page count ─────────────────────────────────────────
    all_pages_info = convert_from_path(pdf_path, dpi=72)
    total_pages = len(all_pages_info)
    del all_pages_info
    print(f'    Total pages : {total_pages}')

    # ── B. Target page ────────────────────────────────────────
    target_pg = PAGE_MAP.get(stem_lower)
    if target_pg is None or target_pg > total_pages:
        target_pg = random.randint(1, total_pages)
        print(f'    Random page : {target_pg}')
    else:
        print(f'    Target page : {target_pg}')

    # ── C. Render page ────────────────────────────────────────
    print(f'    Rendering at {DPI} DPI...')
    page_imgs = convert_from_path(
        pdf_path, dpi=DPI, first_page=target_pg, last_page=target_pg
    )
    page_img = page_imgs[0]
    print(f'    Image size  : {page_img.size[0]}×{page_img.size[1]} px')

    # ── D. Layout detection ───────────────────────────────────
    print(f'    Running DocLayout-YOLO...')
    detections = run_layout_detection(page_img, conf=0.2)
    label_counts = Counter(d['label'] for d in detections)
    print(f'    Detections  : {len(detections)}  →  {dict(label_counts)}')

    # ── E. Visualise ──────────────────────────────────────────
    visualise_detections(
        page_img, detections,
        title=f'{filename}  |  Page {target_pg}/{total_pages}  — DocLayout-YOLO'
    )

    # ── F. Tesseract OCR ──────────────────────────────────────
    text_count = sum(1 for d in detections if d['label'] in TEXT_CLASSES)
    skip_count = len(detections) - text_count
    print(f'    Text regions to OCR        : {text_count}')
    print(f'    Skipped (table/fig/abandon): {skip_count}')
    print(f'    Running Tesseract...\n')

    print(f'{"─"*65}')
    print(f'  EXTRACTED TEXT | {filename} | Page {target_pg}')
    print(f'{"─"*65}')

    ocr_lines = ocr_detections(page_img, detections)

    if not ocr_lines:
        print('  [No text extracted — try lowering conf or increasing DPI]')
    else:
        for label, text in ocr_lines:
            prefix = '### ' if label == 'title' else ''
            print(prefix + text)
            print()   # blank line between regions for readability

    print(f'{"─"*65}')

print('\n✅ Done! Compare output against your PDF pages.')

## Step 7 — Re-run on Any Specific Page

In [ ]:
TARGET_PDF_IDX = 0   # index into pdf_paths
TARGET_PAGE    = 1   # 1-based
CONF_THRESHOLD = 0.2

pdf2   = pdf_paths[TARGET_PDF_IDX]
fname2 = Path(pdf2).name

imgs2 = convert_from_path(pdf2, dpi=DPI, first_page=TARGET_PAGE, last_page=TARGET_PAGE)
img2  = imgs2[0]

dets2 = run_layout_detection(img2, conf=CONF_THRESHOLD)
print(f'Detections on page {TARGET_PAGE}: {dict(Counter(d["label"] for d in dets2))}')

visualise_detections(img2, dets2,
    title=f'{fname2}  |  Page {TARGET_PAGE}  (conf≥{CONF_THRESHOLD})')

print(f'\n--- EXTRACTED TEXT | {fname2} | Page {TARGET_PAGE} ---\n')
for label, text in ocr_detections(img2, dets2):
    print(('### ' if label == 'title' else '') + text)
    print()
print('--- END ---')

## Step 8 — Confidence Threshold Sweep

In [ ]:
TEST_PDF  = pdf_paths[0]
TEST_PAGE = 1

imgs_t = convert_from_path(TEST_PDF, dpi=DPI, first_page=TEST_PAGE, last_page=TEST_PAGE)
img_t  = imgs_t[0]

print(f'Conf sweep | {Path(TEST_PDF).name} p.{TEST_PAGE}\n')
print(f'{"Conf":>6}  {"Total":>6}  Label breakdown')
print('─' * 60)
for conf in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    dets = run_layout_detection(img_t, conf=conf)
    counts = dict(Counter(d['label'] for d in dets))
    print(f'{conf:>6.2f}  {len(dets):>6}  {counts}')

## Step 9 — PSM Tuning per Region (Advanced)

If a specific region type is giving bad output, test different PSM modes on it here.

| PSM | Best for |
|-----|----------|
| 6 | Block of text (paragraphs) |
| 7 | Single line (titles) |
| 11 | Sparse text, no order |
| 3 | Auto (fully automatic, default) |

In [ ]:
# Pick any detection from the last processed page and test all PSM modes
# Change det_index to target a different region
TEST_DET_INDEX = 0   # index into detections list (sorted top→bottom)

if 'detections' not in dir() or not detections:
    print('Run Step 6 or 7 first to populate detections')
else:
    text_dets = [d for d in detections if d['label'] in TEXT_CLASSES]
    if not text_dets:
        print('No text-class detections on last processed page')
    else:
        det = text_dets[min(TEST_DET_INDEX, len(text_dets)-1)]
        crop = crop_detection(page_img, det)

        print(f'Region: {det["label"]}  |  bbox: {det["xyxy"]}\n')

        # Show the crop
        plt.figure(figsize=(8, 3))
        plt.imshow(crop); plt.axis('off')
        plt.title(f'Crop: {det["label"]}')
        plt.tight_layout(); plt.show()

        print(f'{"PSM":>4}  Output')
        print('─' * 60)
        for psm in [3, 6, 7, 11, 12]:
            config = f'--oem 3 --psm {psm} -l eng'
            out = pytesseract.image_to_string(crop, config=config).strip()
            out_preview = out[:120].replace('\n', ' ')
            print(f'{psm:>4}  {out_preview}')